# Stage 1: Per-Wallet Copy Sizing (tier3@2-0)

Fit the wallet universe and per-wallet copy weights ``alpha_w`` on **all data
resolved before August** (the full 2026 history), pick sizing hyperparameters
on **June+July** by sim Sharpe, single **test** pass on **August**.
Copy qty is capped by the reconstructed share-depth ``bucket_avail_copy_qty``.

**Output:** `stage1_scaled_result.json` + `signal_lab/wallet_scaling_{sim,ci,contrib}.csv`



In [59]:
# Setup: imports, paths, constants
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

NB_DIR = Path.cwd() if "__file__" not in globals() else Path(__file__).resolve().parent
sys.path.insert(0, str(NB_DIR))
OUT_DIR = NB_DIR / "signal_lab"

import numpy as np
import pandas as pd

from lib import DEFAULT_TAGS
from signal_lab.filters import COPY_DEFAULT, make_strategy_selection
from signal_lab.signal_lib import spearman_rho
from signal_lab.sizing import (
    block_bootstrap_sharpe,
    capital_constrained_sim,
    sizing_sharpe,
)
from signal_lab.stage1 import (
    attach_copy_wallet_metrics,
    candidate_splits_for,
    load_stage1_data,
)
from signal_lab.wallet_scaling import (
    alpha_kelly,
    alpha_tier,
    attach_depth_cap,
    run_sim,
    sim_row,
    wallet_daily_pnl,
    wallet_stats,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

BUDGET = 10_000.0
EXPOSURE_BUDGET = 10000.0  # per-token (condition_id + token_id) capital cap
MAX_LEAD_DAYS = 14  # keep only trades within this many days of contract resolution
ALPHA_MAX_GRID = (1,)
TIER_GRID = [(nt, am, amin) for nt in (3,) for am in ALPHA_MAX_GRID for amin in (0.0, 0.25)]
UNIFORM_K_GRID = (1.0,)

# Wallet filter: COPY_DEFAULT or strategy_selection (variant follows tags)
STRATEGY_VARIANT = "politics" if "Politics" in DEFAULT_TAGS else "weather"
COPY_FILTER = make_strategy_selection(STRATEGY_VARIANT)

# Headline strategy for the exposure plot + saved result:
#   "kelly" | "tier" | "uniform" | "copy_all"  -> that scheme's scale-chosen config
#   "best"                                     -> max scale-window Sharpe overall
SAVE_STRATEGY = "best"

# PnL variant: (pnl_col, qty_col)
PNL_VARIANT = ("copyable_pnl", "copyable_qty_5m_100")
if 'Politics' in DEFAULT_TAGS:
    PNL_VARIANT = ("copyable_pnl_20m_100", "copyable_qty_20m_100")

PNL_COL, QTY_COL = PNL_VARIANT
AVAIL_COL = QTY_COL.replace("copyable_qty_", "avail_copy_qty_")

# Windows (markets bucketed by last_condition_trade_ts):
#   fit   = everything resolved before Aug ("the whole 2026") -> wallet
#           selection (cell below recomputes metrics on this window) + alphas
#   scale = June + July -> sizing scheme / hyperparameter grid search
#   test  = August -> single honest pass
SPLIT = {"train_end": "2026-06-01", "val_end": "2026-08-01", "test_start": "2026-08-01"}


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [60]:
df_full, df_train, df_val, df_test, wallet_metrics, hold_metrics = load_stage1_data(tags=DEFAULT_TAGS, **SPLIT, max_lead_days=MAX_LEAD_DAYS)
print(f"df_full: {len(df_full):,}")
print(f"  train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}")


Markets: 2870515
Filtered markets for {'Politics'}: 47898
Loading 16 trade shards...
Total trades loaded: 14,516,063
Unique wallets: 35,686
Date range: 2025-01-01 00:00:59+00:00 -> 2026-08-31 05:57:06+00:00
Lead filter (<= 14d before resolution): 10,945,156 trades
split_data_at_dates: train_end=2026-06-01 val_end=2026-08-01 test_start=2026-08-01
  Train:  4,855,293 trades  (15,178 markets)
  Val:    1,549,600 trades  (5,655 markets)
  Test:     447,260 trades  (2,693 markets)
  Total:  6,852,153 trades  (23,526 markets)
df_full: 6,852,153
  train: 4,855,293  val: 1,549,600  test: 447,260


## Copy universe

Candidate wallets = `COPY_FILTER`, fitted on the fit window (all data resolved
before August).



In [61]:
df_fit = pd.concat([df_train, df_val])  # BUY stream resolved before August
fit_metrics = attach_copy_wallet_metrics(df_fit)
del df_fit
wallets = set(COPY_FILTER(fit_metrics, hold_metrics))
print(f"{COPY_FILTER.name} wallets (fit < {SPLIT['val_end']}): {len(wallets)}")


strategy_selection_politics wallets (fit < 2026-08-01): 49


In [62]:
# fit_metrics[fit_metrics['wallet'] == '0x7c63520c2ca9b336af0c205b9ccf68217bb393d4'] #[['buy_copyable_pnl', 'buy_copyable_roi','estimated_copyable_buy_sharpe','max_copyable_drawdown_to_copyable_pnl', 'copyable_pnl_similarity'  ]]

## Share-depth cap

Cap = stage0 Phase 2's per-bucket max copy quantity (`avail_copy_qty_5m_100`), exported with the processed trades.

In [63]:
splits = candidate_splits_for(df_full, wallets, **SPLIT)
splits = attach_depth_cap(splits, avail_col=AVAIL_COL, qty_col=QTY_COL)
for fr in splits.values():
    fr["token_group"] = fr["condition_id"] + "_" + fr["token_id"]
del df_full, df_train, df_val, df_test

for name in ("train", "val", "test"):
    fr = splits[name]
    capped = (fr["bucket_avail_copy_qty"] < fr[QTY_COL]).mean()
    print(f"{name:5s}: {len(fr):,}  trades_capped_by_depth={capped:.3f}")


split_data_at_dates: train_end=2026-06-01 val_end=2026-08-01 test_start=2026-08-01
  Train:    121,296 trades  (4,049 markets)
  Val:       41,862 trades  (1,072 markets)
  Test:       5,732 trades  (  440 markets)
  Total:    168,890 trades  (5,561 markets)
train: 121,296  trades_capped_by_depth=0.000
val  : 41,862  trades_capped_by_depth=0.000
test : 5,732  trades_capped_by_depth=0.000


## Fit-window per-wallet stats

Per-wallet daily pnl (copyable, alpha=1) over the full pre-August candidate
history with mean/std shrinkage -> Sharpe proxy.



In [64]:
pre_aug = pd.concat([splits["train"], splits["val"]])  # resolved before Aug
fit_daily = wallet_daily_pnl(pre_aug, pnl_col=PNL_COL)
st = wallet_stats(fit_daily, pnl_col=PNL_COL)
print(f"wallets with fit-window daily series: {len(st)}")
st[["mu", "sigma", "n_days", "sharpe_proxy", "total_pnl"]].sort_values(
    "sharpe_proxy", ascending=False
).head(15)


wallets with fit-window daily series: 49


,mu,sigma,n_days,sharpe_proxy,total_pnl
wallet,,,,,
0xc0ff6a9ac424210cf218fda5c5753324c34a9953,3429.3541,6538.2193,22,0.2233,75445.7907
0xd721ee218a84e35c2d4437bf509b2195afd1e215,4060.2990,9958.7015,20,0.2132,81205.9791
0x4da76bbf120899fc10fa6e0aad4bffdd19a7355e,2587.3506,11976.7576,42,0.1613,108668.7264
0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,905.8649,3918.0471,143,0.1593,129538.6806
0x09a5d3b75f908d780f2070ac270eabd6cf83caaa,969.5704,1518.4095,50,0.1349,48478.5199
0x8aaaac8e8523492db360b722af35177f4d4d51df,1282.9163,3775.7103,29,0.1216,37204.5719
0xd4140031e313f8d850740a80d2ee6653c925a4db,949.9672,6313.3625,97,0.1169,92146.8160
0x641b56cd1de69da37e9bfefeddc7b277341c5433,754.1804,3596.1468,78,0.1157,58826.0733
0xd35694160b7bc6f734ab2e95072686aceb1a963d,1379.2046,2212.7695,13,0.0902,17929.6604


## Weight schemes

All benchmarked vs copy-all: shrunk max-Sharpe (Kelly), tier, uniform-k.

In [65]:
schemes = {}
for am in ALPHA_MAX_GRID:
    schemes[f"kelly@{am:g}"] = ("kelly", alpha_kelly(st, am), {"alpha_max": am})
for (nt, am, amin) in TIER_GRID:
    schemes[f"tier{nt}@{am:g}-{amin:g}"] = (
        "tier",
        alpha_tier(st, nt, am, amin),
        {"n_tiers": nt, "alpha_max": am, "alpha_min": amin},
    )
for k in UNIFORM_K_GRID:
    schemes[f"uniform@{k:g}"] = ("uniform", pd.Series(k, index=st.index), {"k": k})
schemes["copy_all"] = ("copy_all", pd.Series(1.0, index=st.index), {})

print(f"schemes: {len(schemes)}")


schemes: 5


## Scheme selection (June+July)

Objective: annualized Sharpe of daily resolution-pnl, $1000 per-token exposure
cap. Best config per scheme carries into the single August test pass.



In [66]:
sim_rows = []
best_per_scheme = {}
for name, (scheme, alpha_map, params) in schemes.items():
    res = run_sim(splits["val"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    row = sim_row(scheme, name, "scale", res)
    sim_rows.append(row)
    key = scheme if scheme != "kelly" else "kelly"
    if key not in best_per_scheme or row["sharpe_daily"] > best_per_scheme[key][2]:
        best_per_scheme[key] = (name, params, row["sharpe_daily"])

sim_df = pd.DataFrame(sim_rows)
sim_df[sim_df["split"] == "scale"].sort_values("sharpe_daily", ascending=False).head(15)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
1,tier,tier3@1-0,scale,21089,187556.8600,0.1909,1.4320,77671.4000,388619.9100
3,uniform,uniform@1,scale,28760,185601.1100,0.2164,1.0460,65284.1000,341965.4100
4,copy_all,copy_all,scale,28760,185601.1100,0.2164,1.0460,65284.1000,341965.4100
2,tier,tier3@1-0.25,scale,27431,197190.3600,0.2078,1.0400,74691.6700,373966.7400
0,kelly,kelly@1,scale,30226,214087.1300,0.2500,1.0040,63733.5700,319925.7900


In [67]:
print("Selected per scheme (by scale-window Sharpe):")
for key, (name, params, scale_sharpe) in best_per_scheme.items():
    print(f"  {key:10s} -> {name:>22s}  scale_sharpe={scale_sharpe:.3f}")

assert SAVE_STRATEGY == "best" or SAVE_STRATEGY in best_per_scheme, (
    f"SAVE_STRATEGY={SAVE_STRATEGY!r} not one of {sorted(best_per_scheme)} | 'best'"
)
if SAVE_STRATEGY == "best":
    best_name = max(best_per_scheme.values(), key=lambda x: x[2])[0]
else:
    best_name = best_per_scheme[SAVE_STRATEGY][0]
print(f"\nHeadline strategy ({SAVE_STRATEGY}): {best_name}")


Selected per scheme (by scale-window Sharpe):
  kelly      ->                kelly@1  scale_sharpe=1.004
  tier       ->              tier3@1-0  scale_sharpe=1.432
  uniform    ->              uniform@1  scale_sharpe=1.046
  copy_all   ->               copy_all  scale_sharpe=1.046

Headline strategy (best): tier3@1-0


## Test: single pass per chosen config

One honest test pass for each scheme's val-chosen config (10bps).

In [68]:
for key, (name, params, _scale_sharpe) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    row = sim_row(schemes[name][0], name, "test", res)
    sim_rows.append(row)

sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(OUT_DIR / "wallet_scaling_sim.csv", index=False)
sim_df[sim_df["split"] == "test"].sort_values("sharpe_daily", ascending=False)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
6,tier,tier3@1-0,test,2163,40946.0300,0.3099,2.7470,3909.0100,30613.6800
7,uniform,uniform@1,test,3976,32262.2800,0.2653,1.9470,3631.9900,28268.3100
8,copy_all,copy_all,test,3976,32262.2800,0.2653,1.9470,3631.9900,28268.3100
5,kelly,kelly@1,test,3976,23721.3900,0.1908,1.3160,3897.0900,24316.7800


## Robustness: cost sweep + bootstrap CI

Cost sweep (0/10/30bps) + 7-day block-bootstrap Sharpe CI on test.

In [69]:
ci_rows = []
for key, (name, params, _) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    point, lo, hi = block_bootstrap_sharpe(res["daily_pnl"], block_size=7, n_iter=1000, seed=42)
    ci_rows.append({
        "design": name,
        "pnl": round(res["net_pnl"], 2),
        "roi_w": round(res["net_pnl"] / res["notional"], 4) if res["notional"] > 0 else np.nan,
        "sharpe_daily": round(sizing_sharpe(res["daily_pnl"], 365.0), 3),
        "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
    })

res_all = capital_constrained_sim(splits["test"], "score1", float("inf"), 1.0,
                                  group_col="token_group", group_budget=EXPOSURE_BUDGET,
                                  pnl_col=PNL_COL, qty_col=QTY_COL)
point, lo, hi = block_bootstrap_sharpe(res_all["daily_pnl"], block_size=7, n_iter=1000, seed=42)
ci_rows.append({
    "design": "copy_all",
    "pnl": round(res_all["net_pnl"], 2),
    "roi_w": round(res_all["net_pnl"] / res_all["notional"], 4) if res_all["notional"] > 0 else np.nan,
    "sharpe_daily": round(sizing_sharpe(res_all["daily_pnl"], 365.0), 3),
    "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
})

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT_DIR / "wallet_scaling_ci.csv", index=False)
ci_df


,design,pnl,roi_w,sharpe_daily,ci_lo,ci_hi
0,kelly@1,23721.3900,0.1908,1.3160,-2.7390,2.9530
1,tier3@1-0,40946.0300,0.3099,2.7470,-1.5440,3.2240
2,uniform@1,32262.2800,0.2653,1.9470,-1.4120,3.6730
3,copy_all,32262.2800,0.2653,1.9470,-1.4120,3.6730
4,copy_all,32262.2800,0.2653,1.9470,-1.4120,3.6730


In [70]:
list(schemes['uniform@1'][1].index.values)

['0x01adea599a865a092b51ee4573581a75fc1390f6',
 '0x048215305cbcf7cc790735bf00119551d75c6b0a',
 '0x0787a58e205a70320db638113fe8da18c8800ca3',
 '0x09a5d3b75f908d780f2070ac270eabd6cf83caaa',
 '0x1c853edd4f57b9e8b52977e04f412407dcf231d4',
 '0x1f9e15f39bbd5d163b0eacf0fbef647ced9e4f1a',
 '0x206191d01c411a5b346d11a2df2d463a356b8ec9',
 '0x238fe1b4a91d47bda4fb418dd4a6a09494983015',
 '0x27b820e5203aa114acc2712e0e1d0ad758abb68c',
 '0x2853240a0f4e9e11a949a5cfa6e0fe953a293482',
 '0x33739438875aee79e98a26f87320d984f0703716',
 '0x40cfb29411d29f4fa0908f2a121297042cccd21d',
 '0x4da76bbf120899fc10fa6e0aad4bffdd19a7355e',
 '0x55d0c13d016624e4b816ea561efd5062b4d1f655',
 '0x5bfa0627d533e29faeac9255cd8150a444d4f779',
 '0x6335b3ad24f6fd814c9bd083e4ab79926904ca2b',
 '0x641b56cd1de69da37e9bfefeddc7b277341c5433',
 '0x6725d56690fbe9ba9f301997cef4b503f1cd1bf9',
 '0x682c92615993fd1f75cdfe101efdc1a8adcb17ae',
 '0x6bc74c392c320cfe10d5be61db978a58c8444ad4',
 '0x7656ed7f597a0a61cd307591db198a42b2a7194b',
 '0x7a90bfdb1

## Test-period exposure & PnL over time

Exposure opens at each BUY (`qty = alpha_w * copyable_qty_5m_100` capped by `bucket_avail_copy_qty`, at `price`) and closes at contract resolution `last_condition_trade_ts` — only for contracts resolved within the test window, so unresolved exposure stays open. PnL shown twice: attributed at trade time (`dt`) and at contract resolution time (`last_condition_trade_ts`, resolved contracts only).

In [78]:
import plotly.graph_objects as go

test = splits["test"].copy()
alpha_map = schemes[best_name][1]
test["alpha_w"] = test["wallet"].map(alpha_map).fillna(1.0)
test["raw_copy_pnl"] = test[PNL_COL]
test["wallet_buy_pnl"] = test["pnl"]
test["res_ts"] = pd.to_datetime(test["last_condition_trade_ts"], utc=True, errors="coerce")

# Run sim to get taken mask — exposure/qty must respect the budget
res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
taken_idx = set(res["taken"].values)
test["taken"] = test.index.isin(taken_idx)
test["qty"] = np.clip(test["alpha_w"] * test[QTY_COL], 0.0, test["bucket_avail_copy_qty"])

# Per-trade sim PnL: only for taken trades
per_share = test[PNL_COL] / test[QTY_COL].replace(0, np.nan)
test["copy_pnl"] = np.where(test["taken"], per_share * test["qty"], 0.0)

taken = test[test["taken"]].copy()
print(f"taken trades: {len(taken):,} / {len(test):,}  sim PnL: {taken["copy_pnl"].sum():,.0f}")

window_end = test["dt"].max()
resolved = test["res_ts"] <= window_end
print(
    f"test trades: {len(test):,}  contracts: {test["condition_id"].nunique():,}  "
    f"resolved by {window_end:%Y-%m-%d}: {int(resolved.sum()):,} trades "
    f"({test.loc[resolved, "condition_id"].nunique():,} contracts)"
)
print(
    f"raw {PNL_COL} sum: {test[PNL_COL].sum():,.0f}  "
    f"wallet pnl sum: {test["wallet_buy_pnl"].sum():,.0f}  "
    f"sim copy_pnl sum: {taken["copy_pnl"].sum():,.0f}"
)

# Exposure (scaled): only from taken trades
taken_resolved = taken["res_ts"] <= window_end
open_ev = pd.DataFrame({
    "ev_dt": taken["dt"],
    "exposure_delta": taken["qty"] * taken["price"],
})
close_ev = pd.DataFrame({
    "ev_dt": taken.loc[taken_resolved, "res_ts"],
    "exposure_delta": -(taken.loc[taken_resolved, "qty"] * taken.loc[taken_resolved, "price"]),
})
events = (
    pd.concat([open_ev, close_ev], ignore_index=True)
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
events["exposure"] = events["exposure_delta"].cumsum()

# Exposure (raw): all trades, unscaled
raw_ev = pd.DataFrame({
    "ev_dt": test["dt"],
    "exposure_delta": test[QTY_COL] * test["price"],
})
raw_close = pd.DataFrame({
    "ev_dt": test.loc[resolved, "res_ts"],
    "exposure_delta": -(test.loc[resolved, QTY_COL] * test.loc[resolved, "price"]),
})
raw_events = (
    pd.concat([raw_ev, raw_close], ignore_index=True)
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
raw_events["exposure"] = raw_events["exposure_delta"].cumsum()

def _cum_pnl(dt_col, pnl_col):
    df = test[[dt_col, pnl_col]].rename(columns={dt_col: "ev_dt", pnl_col: "pnl"})
    df = df.sort_values("ev_dt").reset_index(drop=True)
    df["cum_pnl"] = df["pnl"].cumsum()
    return df

pnl_trade = _cum_pnl("dt", "copy_pnl")
pnl_res = _cum_pnl("res_ts", "copy_pnl")
pnl_raw_trade = _cum_pnl("dt", "raw_copy_pnl")
pnl_raw_res = _cum_pnl("res_ts", "raw_copy_pnl")
pnl_wallet_trade = _cum_pnl("dt", "wallet_buy_pnl")
pnl_wallet_res = _cum_pnl("res_ts", "wallet_buy_pnl")

fig = go.Figure()
fig.add_trace(go.Scatter(x=events["ev_dt"], y=events["exposure"], mode="lines",
    name="exposure (scaled)", line=dict(color="rgba(31,119,180,0.6)")))
fig.add_trace(go.Scatter(x=raw_events["ev_dt"], y=raw_events["exposure"], mode="lines",
    name="exposure (raw)", line=dict(dash="dash", color="rgba(31,119,180,0.6)")))
fig.add_trace(go.Scatter(
    x=pnl_trade["ev_dt"], y=pnl_trade["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} sized (trade time)",
))
fig.add_trace(go.Scatter(
    x=pnl_res["ev_dt"], y=pnl_res["cum_pnl"], mode="lines", line=dict(dash="dash"),
    name=f"cum {PNL_COL} sized (resolution time)",
))
fig.add_trace(go.Scatter(
    x=pnl_raw_trade["ev_dt"], y=pnl_raw_trade["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} raw (trade time)", line=dict(color="rgba(255,127,14,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_raw_res["ev_dt"], y=pnl_raw_res["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} raw (resolution time)", line=dict(dash="dash", color="rgba(255,127,14,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_wallet_trade["ev_dt"], y=pnl_wallet_trade["cum_pnl"], mode="lines",
    name="cum wallet buy pnl (trade time)", line=dict(color="rgba(44,160,28,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_wallet_res["ev_dt"], y=pnl_wallet_res["cum_pnl"], mode="lines",
    name="cum wallet buy pnl (resolution time)", line=dict(dash="dash", color="rgba(44,160,28,0.6)"),
))
fig.update_layout(
    title=f"Test-period exposure & PnL over time — {best_name}",
    xaxis_title="Time",
    yaxis_title="USDC",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()


taken trades: 2,163 / 5,732  sim PnL: 40,946
test trades: 5,732  contracts: 440  resolved by 2026-08-30: 5,625 trades (436 contracts)
raw copyable_pnl_20m_100 sum: 32,433  wallet pnl sum: 68,555  sim copy_pnl sum: 40,946


## Per-wallet contributions

Train alphas vs forward (test) wallet stats.

In [72]:
test_daily = wallet_daily_pnl(splits["test"], pnl_col=PNL_COL)
test_st = test_daily.groupby("wallet")[PNL_COL].agg(
    test_pnl="sum", test_n_days="size"
)
test_sharpe = test_daily.groupby("wallet")[PNL_COL].apply(
    lambda s: (s.mean() / s.std() * np.sqrt(365.0)) if s.std() > 0 and len(s) >= 2 else np.nan
).rename("test_sharpe")

contrib = st.join(test_st, how="outer").join(test_sharpe, how="outer").fillna(0.0)
contrib = contrib[contrib["test_n_days"] > 0]
alpha_cont = schemes[best_per_scheme["kelly"][0]][1]
alpha_tier_cont = schemes[best_per_scheme["tier"][0]][1]
contrib["alpha_kelly"] = contrib.index.map(alpha_cont).fillna(1.0)
contrib["alpha_tier"] = contrib.index.map(alpha_tier_cont).fillna(1.0)
contrib = contrib.reset_index()
contrib["test_roi"] = contrib["test_pnl"] / contrib["total_pnl"].replace(0, np.nan)
contrib.to_csv(OUT_DIR / "wallet_scaling_contrib.csv", index=False)

a = contrib["alpha_kelly"].to_numpy()
ts = contrib["test_sharpe"].to_numpy()
valid = np.isfinite(ts)
rho = spearman_rho(pd.Series(a[valid]), pd.Series(ts[valid])) if valid.sum() > 2 else np.nan
print(f"Spearman(alpha_kelly, wallet test sharpe) = {rho:.4f}  (n={int(valid.sum())})")
contrib[["wallet", "alpha_kelly", "alpha_tier", "test_pnl", "test_sharpe", "test_roi"]].head(15)


Spearman(alpha_kelly, wallet test sharpe) = -0.2718  (n=26)


,wallet,alpha_kelly,alpha_tier,test_pnl,test_sharpe,test_roi
0,0x01adea599a865a092b51ee4573581a75fc1390f6,0.8513,0.9608,0.4576,0.0249,0.0000
1,0x0787a58e205a70320db638113fe8da18c8800ca3,0.7609,0.9608,10.4614,0.0446,0.0010
2,0x1f9e15f39bbd5d163b0eacf0fbef647ced9e4f1a,0.4424,0.0000,942.0586,2.8563,0.3103
3,0x206191d01c411a5b346d11a2df2d463a356b8ec9,1.1353,1.9216,26954.3198,12.5921,0.5098
4,0x238fe1b4a91d47bda4fb418dd4a6a09494983015,0.5203,0.0000,-132.3719,-0.8374,-0.0267
5,0x33739438875aee79e98a26f87320d984f0703716,0.4921,0.0000,309.5549,3.8996,0.0732
6,0x40cfb29411d29f4fa0908f2a121297042cccd21d,1.2677,0.9608,5295.2952,4.2028,0.0828
7,0x6725d56690fbe9ba9f301997cef4b503f1cd1bf9,0.6453,0.0000,634.6513,3.3765,0.0793
8,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,1.3478,0.9608,2729.2678,8.8355,0.1048
9,0x7a90bfdb1c7bbb30ab19fdd48a6b49dede972ff9,0.9333,0.9608,-8.8630,-4.4013,-0.0006


## Save stage 1 result

In [ ]:
import json
from datetime import datetime, timezone

best_params = schemes[best_name][2]

wallet_cols = [
    "wallet", "mu", "sigma", "n_days", "total_pnl", "sharpe_proxy",
    "alpha_kelly", "alpha_tier", "test_pnl", "test_n_days", "test_sharpe", "test_roi",
]
wallet_records = contrib[[c for c in wallet_cols if c in contrib.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

test_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == best_name)].iloc[0]
copy_all_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == "copy_all")].iloc[0]

ttl = 300
if 'Politics' in DEFAULT_TAGS:
    ttl = 20*60

metadata = {
    "type": "scaled_copy",
    "saved_strategy": SAVE_STRATEGY,
    "tags": sorted(DEFAULT_TAGS),
    "quote_ttl": ttl,
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": int((contrib["alpha_tier"] > 0).sum()),
    "n_wallets_total": len(wallets),
    "split_sizes": {k: int(len(v)) for k, v in splits.items()},
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_scale_sharpe": float(max(best_per_scheme.values(), key=lambda x: x[2])[2]),
    "test_performance": {
        "config": best_name,
        "trades": int(test_row["trades"]),
        "pnl": float(test_row["pnl"]),
        "roi_w": float(test_row["roi_w"]),
        "sharpe_daily": float(test_row["sharpe_daily"]),
        "copy_all": {
            "trades": int(copy_all_row["trades"]),
            "pnl": float(copy_all_row["pnl"]),
            "roi_w": float(copy_all_row["roi_w"]),
            "sharpe_daily": float(copy_all_row["sharpe_daily"]),
        },
    },
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = NB_DIR / "stage1_scaled_result.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 scaled result -> {out_path.resolve()}")


Saved stage 1 scaled result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_scaled_result.json


## Price-scaling fill experiment (exploratory)

Test a limit-price entry idea on a **sample** (~1k test contracts, copy-default wallets):
copy each candidate copy-wallet BUY at `limit = price * scale` for
`scale ∈ {1.0, 0.98, 0.95, 0.90}` and give the order a **5-minute window** to fill.

- **Fill rule:** filled iff within `(dt, dt+5min]` any trade on the same
  `(condition_id, token_id)` prints at `price <= limit` with a strictly greater timestamp.
- **Fill price:** exactly the limit price, so
  `pnl = copyable_pnl + copyable_qty_5m_100 * (price - limit)` (same formula/quantity as the
  original `copyable_pnl`); unfilled trades contribute 0.
- **Baseline:** `scale = 1.0` is the market-copy (fill immediately at `price`), so it
  must reproduce `sum(copyable_pnl)` on the sample.


In [74]:
from lib import DEFAULT_TRADES_DIR
from signal_lab.wallet_scaling import price_scale_fill_sim

rng = np.random.RandomState(42)
test_markets = np.sort(splits["test"]["condition_id"].unique())
n_sel = min(1000, len(test_markets))
sel_markets = rng.choice(test_markets, size=n_sel, replace=False)
signals = splits["test"][splits["test"]["condition_id"].isin(sel_markets)].copy()
signals = signals[signals["copyable_qty_5m_100"] > 0]
print(f"test markets: {len(test_markets):,}  sampled: {n_sel:,}")
print(f"candidate BUYs (copyable_qty_5m_100>0) on sample: {len(signals):,}")

_tape_cols = ["condition_id", "token_id", "dt", "avg_price"]
tape_parts = []
for f in sorted(DEFAULT_TRADES_DIR.glob("*.parquet")):
    tp = pd.read_parquet(f, columns=_tape_cols)
    tp = tp[tp["condition_id"].isin(sel_markets)]
    if not tp.empty:
        tape_parts.append(tp.rename(columns={"avg_price": "price"}))
tape = (
    pd.concat(tape_parts, ignore_index=True)
    if tape_parts
    else pd.DataFrame(columns=["condition_id", "token_id", "dt", "price"])
)
print(f"fill tape rows (sampled contracts, both sides): {len(tape):,}")


test markets: 440  sampled: 440
candidate BUYs (copyable_qty_5m_100>0) on sample: 3,565
fill tape rows (sampled contracts, both sides): 597,074


In [75]:
SCALES = (1.0, 0.98, 0.95, 0.90)
sim = price_scale_fill_sim(signals, tape, scales=SCALES, window_minutes=5.0)
base_pnl = float(signals["copyable_pnl"].sum())

summary = (
    sim.groupby("scale")
    .agg(signals=("filled", "size"), fills=("filled", "sum"),
         fill_rate=("filled", "mean"), pnl=("pnl", "sum"))
    .reset_index()
)
summary["pnl_pct_of_market"] = summary["pnl"] / base_pnl * 100 if base_pnl else np.nan
summary["delta_vs_market"] = summary["pnl"] - base_pnl
print(f"market-copy pnl (baseline = sum copyable_pnl): {base_pnl:,.2f}")
summary.round(2)


market-copy pnl (baseline = sum copyable_pnl): 14,159.88


,scale,signals,fills,fill_rate,pnl,pnl_pct_of_market,delta_vs_market
0,0.9000,3565,532,0.1500,1809.5000,12.7800,-12350.3800
1,0.9500,3565,811,0.2300,12161.2800,85.8900,-1998.6000
2,0.9800,3565,1053,0.3000,10208.0800,72.0900,-3951.8000
3,1.0000,3565,3565,1.0000,14159.8800,100.0000,0.0000


In [76]:
pw_pnl = sim.pivot_table(index="wallet", columns="scale", values="pnl", aggfunc="sum")
pw_fill = sim.pivot_table(index="wallet", columns="scale", values="filled", aggfunc="mean")
pw = pw_pnl.join(pw_fill.rename(columns={c: f"fill_{c:g}" for c in pw_fill.columns}))
pw = pw.reindex(pw[1.0].sort_values(ascending=False).index)
pw.round(1).head(15)


scale,0.9000,0.9500,0.9800,1.0000,fill_0.9,fill_0.95,fill_0.98,fill_1
wallet,,,,,,,,
0x206191d01c411a5b346d11a2df2d463a356b8ec9,3173.5000,12788.9000,14271.0000,20478.5000,0.3000,0.3000,0.4000,1.0000
0xb43699cbbb52520952833c10737bc43e7625bb3c,3153.7000,3120.3000,3628.2000,3218.4000,0.2000,0.2000,0.3000,1.0000
0xfc2f4f50ce2f6045d35558a5e2d8d4b2ac6610c7,329.3000,2041.3000,1784.1000,1887.9000,0.2000,0.4000,0.5000,1.0000
0x7d51f58c68b427d113d5c84b508161d4e0286f1f,182.2000,821.3000,857.0000,1010.2000,0.1000,0.4000,0.5000,1.0000
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,610.0000,736.2000,763.6000,903.9000,0.2000,0.3000,0.4000,1.0000
0x1f9e15f39bbd5d163b0eacf0fbef647ced9e4f1a,44.5000,334.5000,318.8000,882.5000,0.1000,0.1000,0.1000,1.0000
0x33739438875aee79e98a26f87320d984f0703716,30.8000,36.2000,63.2000,323.5000,0.2000,0.4000,0.5000,1.0000
0x6725d56690fbe9ba9f301997cef4b503f1cd1bf9,410.0000,326.2000,244.0000,295.0000,0.3000,0.4000,0.5000,1.0000
0xa9f2d6b4a8b0b322ddb2f837c2852c1b47056c8e,-35.0000,-77.9000,-149.9000,225.6000,0.1000,0.1000,0.2000,1.0000


In [77]:
sim.to_csv(OUT_DIR / "price_scale_sim.csv", index=False)
summary.round(4).to_csv(OUT_DIR / "price_scale_summary.csv", index=False)
pw.round(2).reset_index().to_csv(OUT_DIR / "price_scale_wallets.csv", index=False)
print("saved -> signal_lab/price_scale_{sim,summary,wallets}.csv")


saved -> signal_lab/price_scale_{sim,summary,wallets}.csv
